# 💳 Credit Score Predictor — Mini ML Project
**Goal:** Predict whether a customer is a credit risk (good/bad) using financial features.  
**Algorithms:** Logistic Regression, Random Forest, Comparison  
**Domain:** FinTech — directly extends the CreditHealth web app (HackMatrix project)

> This project demonstrates end-to-end ML thinking: problem framing → data → preprocessing → modeling → evaluation → business interpretation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 1000

# Synthetic credit dataset (mirrors real-world financial features)
age             = np.random.randint(21, 65, n)
income          = np.random.normal(50000, 20000, n).clip(15000, 200000)
loan_amount     = np.random.normal(15000, 8000, n).clip(1000, 60000)
loan_tenure     = np.random.randint(12, 84, n)
existing_loans  = np.random.randint(0, 5, n)
emi_income_ratio = loan_amount / (income / 12)
employment_type = np.random.choice(['Salaried', 'Self-Employed', 'Business'], n, p=[0.6, 0.25, 0.15])
credit_history  = np.random.choice([0, 1], n, p=[0.25, 0.75])  # 0=bad, 1=good

# Create target: 1=good credit, 0=bad credit (realistic rules)
score = (
    0.3 * (income / income.max()) +
    0.2 * credit_history +
    0.15 * (1 - emi_income_ratio / emi_income_ratio.max()) +
    0.15 * (1 - existing_loans / 5) +
    0.1 * ((age - 21) / 44) +
    np.random.normal(0, 0.05, n)
)
target = (score > score.median()).astype(int)  # balanced split

df = pd.DataFrame({
    'Age': age, 'Income': income.round(0), 'LoanAmount': loan_amount.round(0),
    'LoanTenure_Months': loan_tenure, 'ExistingLoans': existing_loans,
    'EMI_Income_Ratio': emi_income_ratio.round(4),
    'EmploymentType': employment_type, 'CreditHistory': credit_history,
    'GoodCredit': target
})

print("Dataset Shape:", df.shape)
print("\nClass Balance:")
print(df['GoodCredit'].value_counts())
print(f"\nGood Credit Rate: {df['GoodCredit'].mean()*100:.1f}%")
df.head()


## Step 1 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Income by credit
axes[0,0].hist(df[df['GoodCredit']==0]['Income'], bins=30, alpha=0.6, color='#E74C3C', label='Bad Credit')
axes[0,0].hist(df[df['GoodCredit']==1]['Income'], bins=30, alpha=0.6, color='#2ECC71', label='Good Credit')
axes[0,0].set_title('Income by Credit Quality')
axes[0,0].legend()

# EMI ratio
axes[0,1].hist(df[df['GoodCredit']==0]['EMI_Income_Ratio'], bins=30, alpha=0.6, color='#E74C3C', label='Bad Credit')
axes[0,1].hist(df[df['GoodCredit']==1]['EMI_Income_Ratio'], bins=30, alpha=0.6, color='#2ECC71', label='Good Credit')
axes[0,1].set_title('EMI/Income Ratio by Credit Quality')
axes[0,1].legend()

# Existing loans
sns.countplot(x='ExistingLoans', hue='GoodCredit', data=df, ax=axes[0,2],
              palette={0:'#E74C3C', 1:'#2ECC71'})
axes[0,2].set_title('Existing Loans vs Credit Quality')
axes[0,2].legend(['Bad Credit','Good Credit'])

# Credit history
sns.countplot(x='CreditHistory', hue='GoodCredit', data=df, ax=axes[1,0],
              palette={0:'#E74C3C', 1:'#2ECC71'})
axes[1,0].set_title('Credit History vs Credit Quality')
axes[1,0].set_xticklabels(['Bad History','Good History'])
axes[1,0].legend(['Bad Credit','Good Credit'])

# Employment type
ct = df.groupby(['EmploymentType','GoodCredit']).size().unstack()
ct.plot(kind='bar', ax=axes[1,1], color=['#E74C3C','#2ECC71'], edgecolor='black')
axes[1,1].set_title('Employment Type vs Credit Quality')
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)
axes[1,1].legend(['Bad Credit','Good Credit'])

# Age
axes[1,2].hist(df[df['GoodCredit']==0]['Age'], bins=20, alpha=0.6, color='#E74C3C', label='Bad Credit')
axes[1,2].hist(df[df['GoodCredit']==1]['Age'], bins=20, alpha=0.6, color='#2ECC71', label='Good Credit')
axes[1,2].set_title('Age by Credit Quality')
axes[1,2].legend()

plt.suptitle('Credit Score EDA — Feature vs Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 2 — Preprocessing

In [ ]:
# Encode employment type
le = LabelEncoder()
df['EmploymentType_enc'] = le.fit_transform(df['EmploymentType'])

features = ['Age','Income','LoanAmount','LoanTenure_Months','ExistingLoans',
            'EMI_Income_Ratio','CreditHistory','EmploymentType_enc']

X = df[features]
y = df['GoodCredit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


## Step 3 — Train & Compare Models

In [ ]:
models = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':         DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
print(f"{'Model':<25} {'Accuracy':>10} {'AUC-ROC':>10} {'F1':>10}")
print("=" * 60)

for name, m in models.items():
    m.fit(X_train_s, y_train)
    pred  = m.predict(X_test_s)
    proba = m.predict_proba(X_test_s)[:, 1]
    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)
    from sklearn.metrics import f1_score
    f1  = f1_score(y_test, pred)
    results[name] = {'acc': acc, 'auc': auc, 'f1': f1, 'pred': pred, 'proba': proba}
    print(f"{name:<25} {acc*100:>9.2f}% {auc:>10.4f} {f1:>10.4f}")


## Step 4 — ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['#E74C3C','#3498DB','#2ECC71','#F39C12']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={res['auc']:.3f})")

plt.plot([0,1],[0,1],'k--',label='Random (0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Credit Score Prediction', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Step 5 — Feature Importance (Random Forest)

In [ ]:
rf = models['Random Forest']
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'],
         color='#3498DB', edgecolor='black')
plt.title('Feature Importance — Random Forest Credit Predictor', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## Step 6 — Business Interpretation

In [ ]:
best_model = models['Random Forest']
y_pred_final = best_model.predict(X_test_s)
y_proba_final = best_model.predict_proba(X_test_s)[:, 1]

cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

print("=" * 50)
print("    CREDIT RISK MODEL — BUSINESS SUMMARY")
print("=" * 50)
print(f"  Model: Random Forest Classifier")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_final)*100:.2f}%")
print(f"  AUC-ROC   : {roc_auc_score(y_test, y_proba_final):.4f}")
print()
print(f"  Correctly identified GOOD customers : {tp}")
print(f"  Correctly rejected  BAD  customers  : {tn}")
print(f"  Wrongly approved    BAD  customers  : {fp}  ← Risky!")
print(f"  Wrongly rejected    GOOD customers  : {fn}  ← Lost revenue")
print("=" * 50)
print()
print("Top 3 most important factors for credit decision:")
top3 = importance_df.tail(3)['Feature'].values[::-1]
for i, f in enumerate(top3, 1):
    print(f"  {i}. {f}")


## Summary

This mini-project demonstrates:
- End-to-end ML pipeline: data → EDA → preprocessing → modeling → evaluation
- Comparison of 4 algorithms (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting)
- Business interpretation of model results (false positives = financial risk)
- Feature importance analysis

**This directly extends my HackMatrix CreditHealth project** — the web app uses rule-based logic, while this notebook shows the ML-powered version using real predictive models.

**Next steps:** Hyperparameter tuning, SHAP values for explainability, deployment via Flask API
